# 22 · Contextual Retrieval

**Idea:** a chunk ripped out of its document loses context — *"It reduces latency"* — what does "it" refer to? Contextual retrieval (Anthropic, 2024) prepends a short, LLM-generated blurb that situates each chunk inside its parent document *before* embedding. The chunk now carries enough context to be retrieved on its own, measurably cutting failed retrievals.

> Context generation is one call per chunk, so it defaults to Ollama. One cell optionally uses Anthropic — skips if no key.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

Take one document and chunk it small — small chunks are exactly the ones that lose their referents.

In [ ]:
from rag_lab import NOTES_DIR, recursive_chunks

document = (NOTES_DIR / 'hybrid_and_reranking.md').read_text(encoding='utf-8')
plain = recursive_chunks(document, source='hybrid_and_reranking.md', chunk_size=90, overlap=10)
print(f'{len(plain)} plain chunks; example:\n', plain[3].text)

Add context. `contextualize_chunks` prefixes each chunk with an LLM-written line and keeps the raw text in `meta['original']` for citations.

In [ ]:
from rag_lab import contextualize_chunks
from study_buddy import CostTracker

tracker = CostTracker()
contextual = contextualize_chunks(plain, document, provider='ollama', tracker=tracker)

c = contextual[3]
print('context line:', c.meta['context'])
print('\nembedded text:\n', c.text)
print('\ncost:', tracker.summary())

Compare retrieval on an **ambiguous** query. Plain chunks may miss it; contextualized chunks carry the surrounding topic, so the right passage surfaces.

In [ ]:
from rag_lab import NumpyStore

plain_store = NumpyStore(provider='ollama'); plain_store.add(plain)
ctx_store = NumpyStore(provider='ollama'); ctx_store.add(contextual)

query = 'when does it run and why is that order important?'
print('plain:')
for h in plain_store.search(query, k=2):
    print(f'  {h.score:.3f}  {h.chunk.text[:80]}...')
print('contextual:')
for h in ctx_store.search(query, k=2):
    print(f'  {h.score:.3f}  {h.chunk.meta.get("original", h.chunk.text)[:80]}...')

Optionally generate the context lines with Anthropic. Skips if no key.

In [ ]:
try:
    cloud = CostTracker()
    out = contextualize_chunks(plain[:2], document, provider='anthropic', tracker=cloud)
    for c in out:
        print('-', c.meta['context'])
    print('cost:', cloud.summary())
except Exception as e:
    print('anthropic skipped:', type(e).__name__, e)

**Takeaway:** contextual retrieval trades one cheap LLM call per chunk at ingestion time for higher recall at every query — often the best-value fix for "the right chunk exists but never gets retrieved." Keep the original text for display; embed the contextualized version.

Next up → `23_sentence_window.ipynb`, another precision/context trade solved at retrieval time instead of ingestion time.